# Milestone 3 — Text Vectorization, Embeddings, Semantic Search

Part (a): Sentiment classification — TF-IDF baseline vs Sentence Embeddings.
Part (b): Embedding-based **semantic search** over product descriptions.
**Stretch goal:** t-SNE visualization of the embedding space.

In [ ]:
import os, sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, classification_report)
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = '/content/drive/MyDrive/smart-product-intelligence'

## 1. Load reviews and build binary sentiment dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

reviews = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'reviews.csv'))

def to_binary(df):
    df = df[df['rating'].isin([1,2,4,5])].copy()
    df['sentiment'] = (df['rating'] >= 4).astype(int)
    df = df.dropna(subset=['text'])
    df = df[df['text'].astype(str).str.len() > 10]
    return df

train_r = to_binary(reviews[reviews['split']=='train'])
test_r = to_binary(reviews[reviews['split']=='test'])

train_sample = train_r.sample(n=min(20000, len(train_r)), random_state=42)
test_sample = test_r.sample(n=min(3000, len(test_r)), random_state=42)

X_train = train_sample['text'].astype(str).values
y_train = train_sample['sentiment'].values
X_test = test_sample['text'].astype(str).values
y_test = test_sample['sentiment'].values

print(f'Train: {len(X_train)} ({100*y_train.mean():.1f}% pos)')
print(f'Test:  {len(X_test)} ({100*y_test.mean():.1f}% pos)')

## 2. Part (a) — Baseline: TF-IDF + Logistic Regression

In [ ]:
t0 = time.time()
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

lr_tfidf = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = lr_tfidf.predict(X_test_tfidf)
time_tfidf = time.time() - t0

acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
f1_tfidf = f1_score(y_test, y_pred_tfidf, average='macro')
print(f'TF-IDF: accuracy={acc_tfidf:.3f}, macro-F1={f1_tfidf:.3f}, time={time_tfidf:.1f}s')

## 3. Part (a) — Sentence Embeddings + LogReg

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

t0 = time.time()
X_train_emb = embedder.encode(X_train.tolist(), batch_size=64, show_progress_bar=True)
X_test_emb = embedder.encode(X_test.tolist(), batch_size=64, show_progress_bar=True)

lr_emb = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_emb.fit(X_train_emb, y_train)
y_pred_emb = lr_emb.predict(X_test_emb)
time_emb = time.time() - t0

acc_emb = accuracy_score(y_test, y_pred_emb)
f1_emb = f1_score(y_test, y_pred_emb, average='macro')
print(f'Embeddings: accuracy={acc_emb:.3f}, macro-F1={f1_emb:.3f}, time={time_emb:.1f}s')

## 4. Honest finding — TF-IDF wins

| Model | Accuracy | Macro-F1 | Time |
|---|---|---|---|
| **TF-IDF + LogReg** (baseline) | **0.915** | **0.864** | **2.2s** |
| Embeddings + LogReg | 0.885 | 0.826 | 16.1s |

**Bag-of-words actually beats sentence embeddings here.** Reasons:

1. Toy reviews are dominated by simple lexical cues ("great", "broke",
   "love it"). TF-IDF captures these directly; embeddings dilute them.
2. Embeddings shine when **paraphrase invariance** matters — i.e. the
   semantic search use case in Part (b), not for binary classification.
3. TF-IDF is ~7× faster.

The brief asks for evidence-based justification — and the evidence here
says "don't pick the fancier model by default".

In [ ]:
from IPython.display import Image as IPyImage
IPyImage(filename=os.path.join(PROJECT_ROOT, 'figures', '05_m3_results.png'))

## 5. Part (b) — Semantic search demo

In [ ]:
product_embeddings = np.load(os.path.join(PROJECT_ROOT, 'm3_product_embeddings.npy'))
products_idx = pd.read_csv(os.path.join(PROJECT_ROOT, 'm3_product_index.csv'))
print(f'Loaded {product_embeddings.shape[0]:,} embeddings of dim {product_embeddings.shape[1]}')

def find_similar(query, top_k=5):
    q_vec = embedder.encode([query], convert_to_numpy=True)
    sims = cosine_similarity(q_vec, product_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(round(float(sims[i]), 3), products_idx.iloc[i]['title'][:80])
            for i in top_idx]

for q in ['educational puzzle for toddlers',
          'remote control car for boys',
          'soft cuddly teddy bear']:
    print(f'\nQuery: "{q}"')
    for sim, title in find_similar(q, top_k=3):
        print(f'  [{sim:.2f}] {title}')

## 6. Stretch goal — t-SNE visualization

In [ ]:
IPyImage(filename=os.path.join(PROJECT_ROOT, 'figures', '04_m3_embedding_tsne.png'))

## 7. Summary

- TF-IDF baseline is **slightly stronger** than sentence embeddings for
  binary sentiment, and **7× faster**.
- Embeddings prove their worth in Part (b): they power semantic search
  over ~15,000 products.
- t-SNE shows that pretrained sentence encoders capture category
  structure out of the box.

In M4 we move to a full transformer (DistilBERT) fine-tuned on this
same sentiment task.